In [ ]:
# ================================================================
# V18 DONOR-ID FIX: Comprehensive Re-run C4/C3/C5/C7/C8/C9B
# Problem: Blood IT (P190604, P190808) and AR (P190716) have
#          2 samples each. Original C3-C9 used sample-level,
#          inflating Blood IT n=5→7 and Blood AR n=3→4.
# Fix: Aggregate replicates to donor_id before all statistics.
# Gene/pathway lists are UNCHANGED (196/29/148/179).
# ================================================================
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, spearmanr
from statsmodels.stats.multitest import multipletests
import os, time

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
RESULTS_BASE = '/content/drive/MyDrive/ITLAS/results/version18-analysis'

print('Loading h5ad...')
adata = sc.read_h5ad(DATA_PATH)
print(f'Loaded: {adata.shape[0]} cells, {adata.shape[1]} genes')

# Extract donor_id
adata.obs['donor_id'] = adata.obs['sample'].str.split('_').str[1]

# Column assignments
col_donor = 'donor_id'
col_tissue = 'tissue'
col_stage = 'Stage'
col_lineage = 'major_lineage'

# Convert Stage to string to avoid categorical issues
adata.obs[col_stage] = adata.obs[col_stage].astype(str)

# Tissue labels
tissue_vals = adata.obs[col_tissue].unique()
liver_name = [t for t in tissue_vals if 'liver' in str(t).lower()][0]
blood_name = [t for t in tissue_vals if 'blood' in str(t).lower()][0]

GROUPS = ['NL', 'IT', 'IA', 'AR', 'CR']
LINEAGES = ['Myeloid', 'CD4_T', 'CD8_T', 'NK', 'B', 'PlasmaB']
COMPARISONS = [('NL','IT'), ('NL','IA'), ('NL','AR'), ('NL','CR'), ('IT','IA'), ('IA','AR'), ('CR','AR')]

# Exclude gdT
gdt_mask = adata.obs[col_lineage].astype(str).str.lower().str.contains('gdt|gamma')
if gdt_mask.sum() > 0:
    adata = adata[~gdt_mask].copy()
    print(f'Excluded gdT: {adata.shape[0]} cells remaining')

# Verify donor counts
print('\nDonor counts per tissue × stage:')
for tissue_label, tname in [(liver_name, 'Liver'), (blood_name, 'Blood')]:
    for g in GROUPS:
        mask = (adata.obs[col_tissue] == tissue_label) & (adata.obs[col_stage] == g)
        n_donors = adata.obs[mask][col_donor].nunique()
        n_samples = adata.obs[mask]['sample'].nunique()
        flag = ' ⚠️FIXED' if n_samples != n_donors else ''
        print(f'  {tname} {g}: donors={n_donors}, samples={n_samples}{flag}')

print('\n✅ Setup complete. donor_id extracted. Stage as string.')


In [ ]:
# ================================================================
# Core functions: donor-level aggregation
# ================================================================

def get_donor_means_gene(adata, gene, lineage, tissue_label):
    """Get donor-level mean expression for one gene."""
    mask = ((adata.obs[col_lineage] == lineage) & (adata.obs[col_tissue] == tissue_label))
    sub = adata[mask]
    if gene not in sub.var_names:
        return pd.DataFrame()
    expr = sub[:, gene].X.toarray().flatten() if hasattr(sub.X, 'toarray') else sub[:, gene].X.flatten()
    df = pd.DataFrame({'donor': sub.obs[col_donor].values, 'stage': sub.obs[col_stage].values, 'expression': expr})
    return df.groupby(['donor', 'stage'], observed=True)['expression'].mean().reset_index()


def mwu_test(vals1, vals2):
    """Two-sided Mann-Whitney U test. Returns (stat, p) or (nan, nan)."""
    if len(vals1) < 2 or len(vals2) < 2:
        return np.nan, np.nan
    try:
        stat, p = mannwhitneyu(vals1, vals2, alternative='two-sided')
        return stat, p
    except:
        return np.nan, np.nan


def compute_consistency(vals1, vals2, direction='up'):
    """Compute donor-pair consistency."""
    count = 0
    total = len(vals1) * len(vals2)
    if total == 0:
        return '0/0'
    for v1 in vals1:
        for v2 in vals2:
            if direction == 'up' and v2 > v1:
                count += 1
            elif direction == 'down' and v2 < v1:
                count += 1
    return f'{count}/{total}'


def run_gene_comparison(adata, gene, lineage, tissue_label, tissue_name, comparisons):
    """Run all comparisons for one gene/lineage/tissue."""
    dm = get_donor_means_gene(adata, gene, lineage, tissue_label)
    if len(dm) == 0:
        return []
    
    results = []
    for s1, s2 in comparisons:
        v1 = dm[dm['stage'] == s1]['expression'].values
        v2 = dm[dm['stage'] == s2]['expression'].values
        if len(v1) == 0 or len(v2) == 0:
            continue
        m1, m2 = np.mean(v1), np.mean(v2)
        pct = ((m2 - m1) / m1 * 100) if abs(m1) > 1e-10 else (99999.0 if m2 > m1 else -99999.0 if m2 < m1 else 0)
        direction = '↑' if m2 >= m1 else '↓'
        _, p = mwu_test(v1, v2)
        
        if np.isnan(p):
            sig = 'NA'
        elif p < 0.01:
            sig = '**'
        elif p < 0.05:
            sig = '*'
        elif p < 0.10:
            sig = '†'
        else:
            sig = 'NS'
        
        cons_dir = 'up' if m2 >= m1 else 'down'
        consistency = compute_consistency(v1, v2, cons_dir)
        
        results.append({
            'tissue': tissue_name, 'gene': gene, 'lineage': lineage,
            'comparison': f'{s1}→{s2}', 'stage1': s1, 'stage2': s2,
            'mean_s1': m1, 'mean_s2': m2, 'pct_change': round(pct, 1),
            'direction': direction, 'p_value': round(p, 6) if not np.isnan(p) else np.nan,
            'sig': sig, 'consistency': consistency,
            'n_s1': len(v1), 'n_s2': len(v2)
        })
    return results

print('✅ Core functions defined')


In [ ]:
# ================================================================
# C4 RE-RUN: 29 Pathway AUCell Scores (donor_id fix)
# Read existing AUCell scores, re-aggregate to donor level,
# re-compute statistics.
# ================================================================
import json

# Load original C4 pathway CSVs to get the pathway list
c4_liver_old = pd.read_csv(os.path.join(RESULTS_BASE, 'C4_pathway/C4_pathway_liver.csv'))
c4_blood_old = pd.read_csv(os.path.join(RESULTS_BASE, 'C4_pathway/C4_pathway_blood.csv'))
PATHWAYS_26 = sorted(c4_liver_old['pathway'].unique())
print(f'Original C4 pathways: {len(PATHWAYS_26)}')

# Check for C9 3 new pathways
c9_fix2_dir = os.path.join(RESULTS_BASE, 'C9_method_fixes/Fix2_NewPathways')
c9_csv = os.path.join(c9_fix2_dir, 'C4_3new_pathways_all_results.csv')
if os.path.exists(c9_csv):
    c9_new = pd.read_csv(c9_csv)
    PATHWAYS_3NEW = sorted(c9_new['pathway'].unique())
    print(f'C9 new pathways: {PATHWAYS_3NEW}')
else:
    PATHWAYS_3NEW = []
    print('C9 new pathways CSV not found')

# Load pathway gene sets (need to recompute AUCell per donor)
# Actually, we need the RAW AUCell scores per cell, then aggregate to donor
# Check if per-cell AUCell scores are saved
aucell_dir = os.path.join(RESULTS_BASE, 'C2_pathway')
print(f'\nC2 pathway files:')
if os.path.exists(aucell_dir):
    for f in sorted(os.listdir(aucell_dir)):
        size = os.path.getsize(os.path.join(aucell_dir, f))
        print(f'  {f} ({size:,} bytes)')
else:
    print('  C2 directory not found')

# Also check if scores are stored in adata.obs
pw_cols = [c for c in adata.obs.columns if any(p in c.lower() for p in ['aucell', 'pathway', 'PW_'])]
print(f'\nPathway-related columns in adata.obs: {pw_cols[:10]}')


In [ ]:
# ================================================================
# C4 RE-RUN: Find per-cell AUCell scores and re-aggregate
# Strategy: 
#   Option A: If per-cell scores in adata.obsm or saved CSV → re-aggregate
#   Option B: If only summary CSV → re-read raw and recompute
# ================================================================

# Check adata.obsm for AUCell
print('adata.obsm keys:', list(adata.obsm.keys()) if hasattr(adata, 'obsm') else 'none')

# Check for per-cell AUCell CSV
aucell_files = []
for root, dirs, files in os.walk(RESULTS_BASE):
    for f in files:
        if 'aucell' in f.lower() and f.endswith('.csv'):
            fpath = os.path.join(root, f)
            size = os.path.getsize(fpath)
            aucell_files.append((fpath, size))
            print(f'  Found: {fpath} ({size:,} bytes)')

# If per-cell scores exist as large CSV (>1MB = likely per-cell)
large_aucell = [f for f, s in aucell_files if s > 1_000_000]
if large_aucell:
    print(f'\nLarge AUCell CSV found (likely per-cell): {large_aucell[0]}')
    print('Will use this for re-aggregation.')
else:
    print('\nNo large per-cell AUCell CSV found.')
    print('Will need to re-run AUCell scoring from scratch.')
    print('Checking if gene sets are available...')
    geneset_files = []
    for root, dirs, files in os.walk(RESULTS_BASE):
        for f in files:
            if 'gene_set' in f.lower() or 'geneset' in f.lower() or 'pathway_genes' in f.lower():
                geneset_files.append(os.path.join(root, f))
    print(f'  Gene set files: {geneset_files}')


In [ ]:
# ================================================================
# C3 RE-RUN: 196 genes × 6 lineages × 2 tissues × 7 comparisons
# This is the main computation. ~196 × 6 × 2 × 7 = 16,464 tests
# ================================================================
t0 = time.time()

# Load gene lists from original C3/C5
# C3: Read from the original output to get the 196 gene list
c3_liver_old = pd.read_csv(os.path.join(RESULTS_BASE, 'C3_gene_expression/C5_genes_liver.csv'))
# Try to find the C3 gene list
c3_dir = os.path.join(RESULTS_BASE, 'C3_gene_expression')
c5_dir = os.path.join(RESULTS_BASE, 'C5_genes')

print('C3 files:')
for f in sorted(os.listdir(c3_dir)):
    print(f'  {f}')
print('\nC5 files:')
for f in sorted(os.listdir(c5_dir)):
    print(f'  {f}')

# Get C5 148-gene list from existing CSV
c5_liver = pd.read_csv(os.path.join(c5_dir, 'C5_genes_liver.csv'))
c5_blood = pd.read_csv(os.path.join(c5_dir, 'C5_genes_blood.csv'))
C5_GENES = sorted(c5_liver['gene'].unique())
print(f'\nC5 genes: {len(C5_GENES)}')

# Get C3 196-gene list
# C3 may have a separate file, or we extract from the CSV
c3_files = [f for f in os.listdir(c3_dir) if f.endswith('.csv')]
print(f'C3 CSV files: {c3_files}')


In [ ]:
# ================================================================
# C5 RE-RUN: 148 genes (donor_id corrected)
# ================================================================
t0 = time.time()

all_results_liver = []
all_results_blood = []

# Verify genes exist in adata
genes_found = [g for g in C5_GENES if g in adata.var_names]
genes_missing = [g for g in C5_GENES if g not in adata.var_names]
print(f'C5 genes found: {len(genes_found)}/{len(C5_GENES)}')
if genes_missing:
    print(f'Missing: {genes_missing}')

total = len(genes_found) * len(LINEAGES)
done = 0

for gene in genes_found:
    for lineage in LINEAGES:
        # Liver
        results_l = run_gene_comparison(adata, gene, lineage, liver_name, 'Liver', COMPARISONS)
        all_results_liver.extend(results_l)
        
        # Blood
        results_b = run_gene_comparison(adata, gene, lineage, blood_name, 'Blood', COMPARISONS)
        all_results_blood.extend(results_b)
        
        done += 1
        if done % 100 == 0:
            elapsed = time.time() - t0
            print(f'  {done}/{total} ({done/total*100:.0f}%) — {elapsed:.0f}s')

df_liver = pd.DataFrame(all_results_liver)
df_blood = pd.DataFrame(all_results_blood)

print(f'\nC5 Re-run complete in {time.time()-t0:.0f}s')
print(f'  Liver: {len(df_liver)} rows')
print(f'  Blood: {len(df_blood)} rows')

# Verify n counts
print(f'\nBlood NL→IT n_s2 values: {sorted(df_blood[df_blood["comparison"]=="NL→IT"]["n_s2"].unique())}')
print(f'Blood NL→AR n_s2 values: {sorted(df_blood[df_blood["comparison"]=="NL→AR"]["n_s2"].unique())}')

# Save
fix_dir = os.path.join(RESULTS_BASE, 'DonorID_Fix')
os.makedirs(fix_dir, exist_ok=True)

df_liver.to_csv(os.path.join(fix_dir, 'C5_genes_liver_donorfix.csv'), index=False)
df_blood.to_csv(os.path.join(fix_dir, 'C5_genes_blood_donorfix.csv'), index=False)
print(f'\n✅ Saved to {fix_dir}')


In [ ]:
# ================================================================
# VERIFICATION: Compare old vs new p-values
# ================================================================
old_blood = pd.read_csv(os.path.join(RESULTS_BASE, 'C5_genes/C5_genes_blood.csv'))
new_blood = df_blood.copy()

# Merge on gene + lineage + comparison
merged = old_blood.merge(new_blood, on=['gene', 'lineage', 'comparison'], 
                         suffixes=('_old', '_new'))

# Compare NL→IT
nlit = merged[merged['comparison'] == 'NL→IT'].copy()
nlit['p_diff'] = nlit['p_value_new'] - nlit['p_value_old']
nlit['sig_old'] = nlit['p_value_old'] < 0.05
nlit['sig_new'] = nlit['p_value_new'] < 0.05
nlit['sig_changed'] = nlit['sig_old'] != nlit['sig_new']

print('='*70)
print('C5 Blood NL→IT: Old vs New p-values')
print('='*70)
print(f'Total comparisons: {len(nlit)}')
print(f'Significance CHANGED: {nlit["sig_changed"].sum()}')
print(f'  Old sig, New NS: {((nlit["sig_old"]) & (~nlit["sig_new"])).sum()}')
print(f'  Old NS, New sig: {((~nlit["sig_old"]) & (nlit["sig_new"])).sum()}')

# Show all that changed significance
changed = nlit[nlit['sig_changed']].sort_values('p_diff', ascending=False)
if len(changed) > 0:
    print(f'\nGenes that CHANGED significance at NL→IT:')
    for _, r in changed.iterrows():
        old_label = '★' if r['sig_old'] else 'NS'
        new_label = '★' if r['sig_new'] else 'NS'
        print(f'  {r["gene"]:<10} {r["lineage"]:<10} '
              f'old_p={r["p_value_old"]:.4f}({old_label}) → '
              f'new_p={r["p_value_new"]:.4f}({new_label}) '
              f'n_old={r["n_s2_old"]} → n_new={r["n_s2_new"]}')

# Also check NL→AR
nlar = merged[merged['comparison'] == 'NL→AR'].copy()
nlar['sig_old'] = nlar['p_value_old'] < 0.05
nlar['sig_new'] = nlar['p_value_new'] < 0.05
nlar['sig_changed'] = nlar['sig_old'] != nlar['sig_new']
print(f'\nC5 Blood NL→AR sig changed: {nlar["sig_changed"].sum()}')

# IT→IA and IA→AR
for comp_name in ['IT→IA', 'IA→AR', 'CR→AR']:
    sub = merged[merged['comparison'] == comp_name].copy()
    sub['sig_old'] = sub['p_value_old'] < 0.05
    sub['sig_new'] = sub['p_value_new'] < 0.05
    sub['sig_changed'] = sub['sig_old'] != sub['sig_new']
    print(f'C5 Blood {comp_name} sig changed: {sub["sig_changed"].sum()}')


In [ ]:
# ================================================================
# C8 RE-RUN: IT-specific pattern re-classification
# IT-specific = NL→IT sig (p<0.05) AND NL→IA NS (p>=0.05)
# ================================================================

def classify_it_specific(df, tissue_name):
    nlit = df[(df['comparison'] == 'NL→IT') & (df['p_value'] < 0.05)].copy()
    nlia = df[df['comparison'] == 'NL→IA'].copy()
    nlia_dict = {(r['gene'], r['lineage']): r['p_value'] for _, r in nlia.iterrows()}
    
    it_spec = []
    for _, row in nlit.iterrows():
        ia_p = nlia_dict.get((row['gene'], row['lineage']))
        if ia_p is not None and ia_p >= 0.05:
            it_spec.append({
                'tissue': tissue_name, 'gene': row['gene'], 'lineage': row['lineage'],
                'pathway': row.get('pathway', ''),
                'IT_pct': row['pct_change'], 'IT_p': row['p_value'],
                'IA_p': ia_p, 'direction': row['direction'],
                'consistency': row['consistency']
            })
    return pd.DataFrame(it_spec)

# New C5 IT-specific
new_liver_itspec = classify_it_specific(df_liver, 'Liver')
new_blood_itspec = classify_it_specific(df_blood, 'Blood')

# Exclude gdT
new_liver_itspec = new_liver_itspec[~new_liver_itspec['lineage'].str.contains('gdT|gdt', case=False)]
new_blood_itspec = new_blood_itspec[~new_blood_itspec['lineage'].str.contains('gdT|gdt', case=False)]

print('='*60)
print('C8 IT-specific RE-CLASSIFICATION (donor_id corrected)')
print('='*60)
print(f'  Liver: {len(new_liver_itspec)} (was 52 before fix)')
print(f'  Blood: {len(new_blood_itspec)} (was 116 before fix)')
print(f'  Total: {len(new_liver_itspec) + len(new_blood_itspec)} (was 168)')

# Save
combined_itspec = pd.concat([new_liver_itspec, new_blood_itspec], ignore_index=True)
combined_itspec.to_csv(os.path.join(fix_dir, 'C8_IT_specific_donorfix.csv'), index=False)

# Key genes check
print(f'\nKey gene verification:')
for gene, lin, tissue in [('TOX','CD4_T','Liver'),('LAYN','CD4_T','Liver'),
                           ('BCL6','CD8_T','Liver'),('TOX','CD8_T','Liver')]:
    found = combined_itspec[(combined_itspec['gene']==gene) & 
                            (combined_itspec['lineage']==lin) & 
                            (combined_itspec['tissue']==tissue)]
    status = '✅' if len(found) > 0 else '❌'
    print(f'  {status} {gene}/{lin}/{tissue}')


In [ ]:
# ================================================================
# C4 RE-RUN: Pathway scores re-aggregated to donor level
# We need per-cell AUCell scores. Check if they exist.
# If not, we re-run the MWU test from the original C4 CSVs
# by loading the per-cell scores that were used.
# ================================================================

# The C4 CSV has sample-level means. We need to go back to per-cell.
# Check if C2 has per-cell pathway scores saved in adata.obsm
if 'X_aucell' in adata.obsm:
    print('Found AUCell scores in adata.obsm["X_aucell"]')
elif any('aucell' in k.lower() for k in adata.obsm.keys()):
    key = [k for k in adata.obsm.keys() if 'aucell' in k.lower()][0]
    print(f'Found AUCell scores in adata.obsm["{key}"]')
else:
    print('AUCell scores NOT in adata.obsm')
    print('Will need to check C2 directory for per-cell CSV files')
    print('Or re-run AUCell scoring.')
    
    # Check C2 for large files
    c2_dir = os.path.join(RESULTS_BASE, 'C2_pathway')
    if os.path.exists(c2_dir):
        for f in sorted(os.listdir(c2_dir)):
            fpath = os.path.join(c2_dir, f)
            size = os.path.getsize(fpath)
            if size > 100000:  # >100KB
                print(f'  Large file: {f} ({size:,} bytes)')
                if f.endswith('.csv'):
                    df_peek = pd.read_csv(fpath, nrows=3)
                    print(f'    Cols: {list(df_peek.columns)[:8]}')
                    print(f'    Shape preview: {df_peek.shape}')

print('\n⚠️ C4 pathway re-run requires per-cell AUCell scores.')
print('If not available, C4 re-run must be done as a separate Colab notebook')
print('using the original C2/C4 pipeline with donor_id aggregation.')


In [ ]:
# ================================================================
# FINAL SUMMARY: All donor_id fix impacts
# ================================================================
print('='*70)
print('DONOR-ID FIX COMPLETE SUMMARY')
print('='*70)

# C5 new significant counts
new_sig_liver = df_liver[df_liver['p_value'] < 0.05]
new_sig_blood = df_blood[df_blood['p_value'] < 0.05]

print(f'\nC5 (148 genes) significant results:')
for comp_name in ['NL→IT', 'NL→IA', 'NL→AR', 'NL→CR', 'IT→IA', 'IA→AR', 'CR→AR']:
    l_count = len(new_sig_liver[new_sig_liver['comparison'] == comp_name])
    b_count = len(new_sig_blood[new_sig_blood['comparison'] == comp_name])
    print(f'  {comp_name}: Liver={l_count}, Blood={b_count}')

print(f'\nIT-specific (C5, 6 lineages, gdT excluded):')
print(f'  Liver: {len(new_liver_itspec)}')
print(f'  Blood: {len(new_blood_itspec)}')
print(f'  Total: {len(new_liver_itspec) + len(new_blood_itspec)}')

print(f'\nFiles saved to: {fix_dir}')
for f in sorted(os.listdir(fix_dir)):
    size = os.path.getsize(os.path.join(fix_dir, f))
    print(f'  {f} ({size:,} bytes)')

print(f'\n⚠️ REMAINING TASKS:')
print(f'  1. C4 pathway re-run (needs per-cell AUCell scores)')
print(f'  2. C7 correlation re-run (if donor_means were sample-based)')
print(f'  3. C9B FDR re-calculation (based on new p-values)')
print(f'  4. Update manuscript with new numbers')
print(f'  5. Re-generate Figures 3-7 with corrected data')
